# Per-Subject DSAR Erasure · 1. PII column registry

The registry is the **config** that drives the whole erasure engine (the equivalent of the DBA-provided PII
column list in the current AWS flow). One row per **PII column**, declaring:

- `table_name` — which table it lives in
- `column_name` — the physical column
- `identifier_role` — how this column is used to **match** a subject: `first_name`, `last_name`, `email`,
  `full_name` (single "First Last" column), or `none` (it's PII to erase but not used for matching, e.g. `pnr`)
- `erase_strategy` — how to erase it once a row matches: `scalar_token`, `json_key`, or `pnr_token`
- `json_tag` — for `json_key` columns, which compiled mask to apply (`name` / `email`), reusing the masking repo's approach
- `is_identifier` — whether this column participates in the WHERE match

**Matching rule (your decision — most conservative):** a subject matches a row only if **every identifier
column registered for that table matches**. So `customer_profile` (first+last+email) requires all three;
`contact_email_only` (email only) requires just email; `booking` (full_name) matches on the full name string;
`loyalty_split` requires first_nm AND last_nm.
- `subject_scope` — `customer` or `employee`, from the **table-level** `subject_scope` UC tag (untagged → `customer`). The erasure job skips `employee` tables on a customer request.


## 0. Configuration

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema")
dbutils.widgets.text("tag_key", "pii_aa", "3 UC column tag key for PII")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
TAG_KEY = dbutils.widgets.get("tag_key")
FQ      = f"{CATALOG}.{SCHEMA}"
print("Schema:", FQ, "| tag_key:", TAG_KEY)

## 1. Read the PII column tags (the single source of truth)
`00` tagged every PII column with `pii=<type>`. We read those tags from `information_schema.column_tags` —
so **what is PII is declared once, by the tag**, exactly like the masking solution's governed `pii` tag.
Tagging a new column is all it takes to bring it into scope; no code change here.

In [0]:
tagged = spark.sql(f"""
  SELECT table_name, column_name, tag_value AS pii_type
  FROM {CATALOG}.information_schema.column_tags
  WHERE schema_name = '{SCHEMA}' AND tag_name = '{TAG_KEY}'
  ORDER BY table_name, column_name
""").collect()
print(f"found {len(tagged)} PII-tagged columns")
display(spark.createDataFrame(tagged))

## 2. Enrich the tags into the registry
The tag says *what type* of PII a column is (`name` / `email` / `pnr`). The erasure engine needs a little
more that a tag can hold — **how to match** on it and **how to erase** it. We derive that from the pii_type
plus the column's data type (a JSON/string payload masked in-place vs a plain scalar):

| pii_type | scalar column → | STRING JSON payload column → |
|---|---|---|
| `name` (single full-name col like `full_name`) | match `full_name`, redact token | — |
| `name` (paired first/last cols) | match that name part, redact token | — |
| `email` | match email, redact token | — |
| `pnr` | not matched, redact token (`pnr_token`) | — |
| any, inside JSON (`hit_payload`) | — | match on embedded email+first+last, `json_key` mask |

The only per-column hints we can't infer are (a) which name columns are *first* vs *last* vs *full*, and
(b) which column is the JSON payload. We keep a tiny **role map** for those; everything else comes from tags.
Add a table by tagging its columns and, if it has split/again-named name columns, one role-map entry.

In [0]:
# Minimal hints the tag can't express: name-column role + which columns are JSON payloads.
# key = (table, column) -> role in {"first_name","last_name","full_name"}; default for a lone name col.
name_roles = {
    ("customer_profile", "first_name"): "first_name",
    ("customer_profile", "last_name"):  "last_name",
    ("loyalty_split",    "first_nm"):   "first_name",
    ("loyalty_split",    "last_nm"):    "last_name",
    ("booking",          "full_name"):  "full_name",
}
json_payload_cols = {("app_hits_json", "hit_payload")}   # STRING columns holding JSON to mask in place

# Subject scope per table: DSAR/CCPA is a CUSTOMER right, so employee-only sources (e.g. Merlot crew
# tables) must be skipped on a customer request. Scope is declared once as a TABLE-level UC tag
# `subject_scope` = customer | employee; untagged tables default to `customer` (safe for existing tables).
# Tag a table with:  ALTER TABLE <cat>.<sch>.<tbl> SET TAGS ('subject_scope' = 'employee');
SCOPE_TAG = "subject_scope"
scope_by_table = {
    r["table_name"]: (r["tag_value"] or "customer").strip().lower()
    for r in spark.sql(f"""
        SELECT table_name, tag_value
        FROM {CATALOG}.information_schema.table_tags
        WHERE schema_name = '{SCHEMA}' AND tag_name = '{SCOPE_TAG}'
    """).collect()
}
def scope_of(table):
    return scope_by_table.get(table, "customer")

def derive(table, column, pii_type):
    """Return (identifier_role, is_identifier, erase_strategy, json_tag)."""
    if (table, column) in json_payload_cols:
        # matched on embedded identifiers, erased with the compiled JSON mask
        return ("email", True, "json_key", pii_type)
    if pii_type == "pnr":
        return ("none", False, "pnr_token", None)   # PII, erased, but not a match key
    if pii_type == "email":
        return ("email", True, "scalar_token", None)
    if pii_type == "name":
        role = name_roles.get((table, column), "full_name")
        return (role, True, "scalar_token", None)
    # unknown type -> erase as scalar, don't match on it (safe default)
    return ("none", False, "scalar_token", None)

rows = []
for r in tagged:
    role, is_id, strat, jtag = derive(r["table_name"], r["column_name"], r["pii_type"])
    rows.append((r["table_name"], r["column_name"], role, is_id, strat, jtag, scope_of(r["table_name"])))

spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.pii_column_registry (
  table_name      STRING,
  column_name     STRING,
  identifier_role STRING,   -- first_name | last_name | email | full_name | none
  is_identifier   BOOLEAN,  -- participates in the WHERE match?
  erase_strategy  STRING,   -- scalar_token | json_key | pnr_token
  json_tag        STRING,   -- name | email  (only for json_key)
  subject_scope   STRING    -- customer | employee  (from the table-level `subject_scope` tag; default customer)
) COMMENT 'Config derived from the pii column tags + role map + table subject_scope tag. Drives the erasure engine.'
""")

def s(v):
    return "NULL" if v is None else "'" + v.replace("'","''") + "'"
vals = ",\n".join(
    f"({s(t)}, {s(c)}, {s(role)}, {str(i).lower()}, {s(e)}, {s(j)}, {s(scope)})"
    for t,c,role,i,e,j,scope in rows
)
spark.sql(f"INSERT INTO {FQ}.pii_column_registry VALUES\n{vals}")
display(spark.table(f"{FQ}.pii_column_registry"))


## 3. Note on the JSON table's matching
For `app_hits_json` the identifiers are **inside** the JSON string. The engine matches a subject when the
payload contains that subject's email (and, for safety, first & last name) via `LIKE` on the string, then
applies the compiled `json_key` mask to redact `name` + URL `firstName`/`lastName`/`email` for **only those
matched rows** — schema and every non-PII value (`appName`, `revenue`) preserved.

Next: **`02_subject_erasure_engine`**.